In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_squared_error

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print("All imports successful!")
print(f"Pandas: {pd.__version__}")

## Section 1: Load Data & Train/Val Split
Goal: Load processed features and create time-based train/validation split

In [ ]:
# Load processed dataset
df = pd.read_pickle('../data/processed/df_train.pkl')

print(f"Shape:      {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Memory:     {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"\nColumns: {df.columns.tolist()}")

### 1.2 Time-based train/validation split
* Train: everything before last 28 days
* Val:   last 28 days (mimics M5 competition evaluation)

In [ ]:
# M5 competition uses last 28 days as validation
# Our date range ends 2016-04-24
# Val = last 28 days = 2016-03-28 to 2016-04-24

cutoff_date = df['date'].max() - pd.Timedelta(days=28)

train = df[df['date'] <= cutoff_date].copy()
val   = df[df['date'] >  cutoff_date].copy()

print(f"Cutoff date:  {cutoff_date.date()}")
print(f"\nTrain shape:  {train.shape}")
print(f"Train dates:  {train['date'].min().date()} to {train['date'].max().date()}")
print(f"\nVal shape:    {val.shape}")
print(f"Val dates:    {val['date'].min().date()} to {val['date'].max().date()}")
print(f"\nVal days:     {val['date'].nunique()} unique days")
print(f"Val items:    {val['item_id'].nunique()} unique items")

In [ ]:
# Target variable
TARGET = 'sales'

# Columns to exclude from features
EXCLUDE = [
    'id',           # identifier
    'sales',        # target variable
    'date',         # raw date — extracted features already built
    'wm_yr_wk',     # week key — already used in price merge
]

# Feature columns
FEATURES = [c for c in df.columns if c not in EXCLUDE]

print(f"Target:   {TARGET}")
print(f"Features: {len(FEATURES)}")
print(f"\nFeature list:")
for i, f in enumerate(FEATURES):
    print(f"  {i+1:2d}. {f}")

## Section 2: Baseline Models
Goal: Establish benchmark RMSE that LightGBM must beat
Two baselines: naive (last 28 days avg) and moving average

### 2.1 Naive baseline
Predict last 28 days average sales for each item-store
Simplest possible model — LightGBM must beat this

In [ ]:
# Naive baseline — predict mean of last 28 days per item-store
last_28 = train[train['date'] > cutoff_date - pd.Timedelta(days=28)]

naive_pred = last_28.groupby(
    ['item_id', 'store_id']
)['sales'].mean().reset_index()
naive_pred.columns = ['item_id', 'store_id', 'naive_pred']

# Merge predictions onto val set
val = val.merge(naive_pred, on=['item_id', 'store_id'], how='left')
val['naive_pred'] = val['naive_pred'].fillna(0)

# Calculate RMSE
naive_rmse = np.sqrt(mean_squared_error(
    val[TARGET], val['naive_pred']
))

print(f"Naive baseline RMSE: {naive_rmse:.4f}")

### 2.2 Moving average baseline
Predict 7-day rolling average — slightly smarter than naive

In [ ]:
# Moving average baseline — last 7 days of training only
last_7 = train[train['date'] > cutoff_date - pd.Timedelta(days=7)]

ma_pred = last_7.groupby(
    ['item_id', 'store_id']
)['sales'].mean().reset_index()
ma_pred.columns = ['item_id', 'store_id', 'ma_pred']

val = val.merge(ma_pred, on=['item_id', 'store_id'], how='left')
val['ma_pred'] = val['ma_pred'].fillna(0)

ma_rmse = np.sqrt(mean_squared_error(
    val[TARGET], val['ma_pred']
))
print(f"Moving average RMSE: {ma_rmse:.4f}")

### 2.3 Baseline comparison table

In [ ]:
# Summary table
baselines = pd.DataFrame({
    'model': ['Naive (last 28d avg)', 'Moving Average (7d)'],
    'rmse':  [naive_rmse, ma_rmse]
})
baselines['rmse'] = baselines['rmse'].round(4)
baselines = baselines.sort_values('rmse')

print("Baseline results:")
print(baselines.to_string(index=False))
print(f"\nLightGBM target: beat {baselines['rmse'].min():.4f}")

### Key findings — Section 2: Baselines

- Naive baseline RMSE:    2.2186
- Moving average RMSE:    2.0970 (5.5% better than naive)
- LightGBM target:        beat 2.0970
- Context: avg daily sales ~1.13 units — error > mean (intermittent demand challenge)
- Expected LightGBM RMSE: 1.3–1.5 (30–35% improvement over baseline)